In [1]:
import os
import pandas as pd
import numpy as np
import yaml
import pickle

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold


from utils.training_utils import find_specific_variables
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(os.path.join('..', 'data', 'train_test', 'train_encoded.csv'))

print(df.shape)
df.head()

(32940, 11)


,contact,month,age,cons.conf.idx,cons.price.idx,emp.var.rate,euribor3m,nr.employed,pdays,was_contacted_before,y
0,0.0,9.0,31.0,-29.8,92.379,-3.4,0.803,5017.5,999.0,0.0,0
1,1.0,6.0,39.0,-36.4,93.994,1.1,4.857,5191.0,999.0,0.0,0
2,0.0,3.0,34.0,-42.7,93.918,1.4,4.958,5228.1,999.0,0.0,0
3,1.0,6.0,36.0,-36.4,93.994,1.1,4.856,5191.0,999.0,0.0,0
4,0.0,1.0,25.0,-31.4,92.201,-2.9,0.825,5076.2,999.0,0.0,0


In [3]:
features = yaml.safe_load(open(os.path.join('..', 'src', 'config', 'feature_config.yaml'), 'r'))
feature_target = find_specific_variables(features, 'target', specific_value=True)

In [5]:
seletor = pickle.load(
    open(os.path.join('..', 'models', 'encoders', 'seletor_2.pkl'), 'rb')
)

seletor.features

['age',
 'cons.conf.idx',
 'cons.price.idx',
 'contact',
 'emp.var.rate',
 'euribor3m',
 'month',
 'nr.employed',
 'pdays',
 'was_contacted_before']

In [6]:
scale_pos_weight = df[df[feature_target]==0].shape[0] / df[df[feature_target]==1].shape[0]

models = {
    'DT': DecisionTreeClassifier(),
    'RF': RandomForestClassifier(),
    'GBT': GradientBoostingClassifier(),
    'ADA': AdaBoostClassifier(),
    'XGB': XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    'LGBM': LGBMClassifier(),

    'DT_bal': DecisionTreeClassifier(class_weight='balanced'),
    'RF_bal': RandomForestClassifier(class_weight='balanced'),
    'GBT_bal': GradientBoostingClassifier(),
    'ADA_bal': AdaBoostClassifier(),
    'XGB_bal': XGBClassifier(scale_pos_weight=scale_pos_weight, use_label_encoder=False, eval_metric='logloss'),
    'LGBM_bal': LGBMClassifier(class_weight='balanced'),
}

In [7]:
sampling_strategies = {
    'Original': None,
    'Undersampling': RandomUnderSampler(random_state=96),
    'Oversampling': RandomOverSampler(random_state=96),
    'SMOTE': SMOTE(random_state=96)
}

results_sampling = {}

for sampling_name, sampler in sampling_strategies.items():
    print(f'\nSampling Strategy: {sampling_name}')
    results_sampling[sampling_name] = {}
    
    for model_name, model in models.items():
        skf = StratifiedKFold(n_splits=5, random_state=96, shuffle=True)
        
        if sampler is not None:
            pipeline = Pipeline([
                ('sampler', sampler),
                ('classifier', model)
            ])
        else:
            pipeline = model

        scores = cross_val_score(
            pipeline,
            df[seletor.features],
            df[feature_target],
            cv=skf,
            scoring='roc_auc'
        )

        results_sampling[sampling_name][model_name] = scores

        print(f'{model_name}: {np.mean(scores):.4f} +/- {np.std(scores):.4f}')



Sampling Strategy: Original
DT: 0.6095 +/- 0.0088
RF: 0.7427 +/- 0.0093
GBT: 0.7980 +/- 0.0095
ADA: 0.7944 +/- 0.0092
XGB: 0.7878 +/- 0.0050
LGBM: 0.7991 +/- 0.0077
DT_bal: 0.5998 +/- 0.0109
RF_bal: 0.6897 +/- 0.0106
GBT_bal: 0.7980 +/- 0.0095
ADA_bal: 0.7944 +/- 0.0092
XGB_bal: 0.7878 +/- 0.0050
LGBM_bal: 0.7955 +/- 0.0064

Sampling Strategy: Undersampling
DT: 0.6934 +/- 0.0082
RF: 0.7520 +/- 0.0090
GBT: 0.7984 +/- 0.0106
ADA: 0.7913 +/- 0.0105
XGB: 0.7810 +/- 0.0067
LGBM: 0.7929 +/- 0.0092
DT_bal: 0.6909 +/- 0.0079
RF_bal: 0.7515 +/- 0.0097
GBT_bal: 0.7985 +/- 0.0105
ADA_bal: 0.7913 +/- 0.0105
XGB_bal: 0.7810 +/- 0.0067
LGBM_bal: 0.7929 +/- 0.0092

Sampling Strategy: Oversampling
DT: 0.6026 +/- 0.0119
RF: 0.6942 +/- 0.0113
GBT: 0.8005 +/- 0.0088
ADA: 0.7946 +/- 0.0087
XGB: 0.7723 +/- 0.0041
LGBM: 0.7950 +/- 0.0071
DT_bal: 0.6060 +/- 0.0106
RF_bal: 0.6945 +/- 0.0110
GBT_bal: 0.8004 +/- 0.0090
ADA_bal: 0.7946 +/- 0.0087
XGB_bal: 0.7723 +/- 0.0041
LGBM_bal: 0.7950 +/- 0.0071

Sampling 